<a href="https://colab.research.google.com/github/rajidush/RP-J26-DS-316/blob/c3%2Fmodel-tuning/component3_socratic_educator/notebooks/train_inquire_lora_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Train the INQUIRE LoRA adapter (Component 3, ticket 04)

Fine-tunes `google/gemma-3-1b-it` with the **fixed** LoRA config in `inquire_training.py`
(r=16, alpha=32, attention + MLP projections, LR 1e-4, 3 epochs, completion-only loss),
evaluating and saving every epoch and keeping the checkpoint with the lowest validation loss.
There is no hyperparameter search.

**Before you start**
- Runtime → Change runtime type → **T4 GPU**.
- Colab secret `HF_TOKEN` (🔑 in the sidebar) for a Hugging Face account that has accepted the Gemma licence.
- From the repo: `component3_socratic_educator/inquire_training.py` and the four files in
  the `build` folder named by `DATASET_FOLDER` below, e.g. `component3_socratic_educator/evidence/inquire_dataset_2026-10-10/` (`manifest.json`, `train.jsonl`, `validation.jsonl`, `test.jsonl`).

**Outputs** go to Google Drive only (`MyDrive/c3_inquire_training/<run id>/`): checkpoints, the adapter and `run_record.json`.
Only `run_record.json` comes back to the repo, as evidence. The adapter never does.

In [ ]:
!pip -q install "transformers==4.56.2" "trl==0.23.1" "peft==0.17.1" "accelerate==1.10.1" "datasets==4.0.0"

In [ ]:
import time
from google.colab import drive
drive.mount("/content/drive")

RUN_ID = time.strftime("%Y-%m-%d_%H%M", time.gmtime())
RUN_DIR = f"/content/drive/MyDrive/c3_inquire_training/{RUN_ID}"
ADAPTER_DIR = f"{RUN_DIR}/adapter"
DATA_DIR = "/content/inquire_dataset"
DATASET_FOLDER = "inquire_dataset_2026-10-10"   # the repo `build` folder you upload from (recorded in the run record)
print("run directory:", RUN_DIR)

Upload `inquire_training.py`, `manifest.json`, `train.jsonl`, `validation.jsonl` and `test.jsonl` (select all five at once).

In [ ]:
import os
import re
from google.colab import files

EXPECTED = ("inquire_training.py", "manifest.json", "train.jsonl", "validation.jsonl", "test.jsonl")
uploaded = files.upload()
os.makedirs(DATA_DIR, exist_ok=True)
for name in uploaded:
    base = re.sub(r" \(\d+\)(?=\.)", "", name)  # Colab renames re-uploads to 'file (1).jsonl'
    if base not in EXPECTED:
        print("ignored:", name)
        continue
    os.replace(name, base if base == "inquire_training.py" else f"{DATA_DIR}/{base}")
missing = [n for n in EXPECTED if not os.path.exists(n if n.endswith(".py") else f"{DATA_DIR}/{n}")]
assert not missing, f"still missing: {missing}"
print("uploaded:", ", ".join(EXPECTED))

Stop here unless the dataset is exactly the `build` output its manifest describes.

In [ ]:
import inquire_training as tr

manifest = tr.verify_dataset(DATA_DIR)   # raises DatasetError on a missing or mismatched file
MANIFEST_SHA256 = tr.manifest_sha256(DATA_DIR)
for split in ("train", "validation", "test"):
    c = manifest["counts"][split]
    print(f"{split:<10} {c['rows']:>3} rows, {c['scenarios']:>2} scenarios")
print("manifest sha256:", MANIFEST_SHA256)

In [ ]:
import torch
from google.colab import userdata
from huggingface_hub import login

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
GPU = torch.cuda.get_device_name(0)
login(token=userdata.get("HF_TOKEN"))
print("GPU:", GPU, "| precision:", tr.PRECISION)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

set_seed(tr.SEED)
tokenizer = AutoTokenizer.from_pretrained(tr.BASE_MODEL)
# fp32 (see tr.PRECISION_REASON); eager attention is the recommended setting for training Gemma 3.
model = AutoModelForCausalLM.from_pretrained(tr.BASE_MODEL, dtype=torch.float32, attn_implementation="eager")
MODEL_REVISION = getattr(model.config, "_commit_hash", None)
print("model revision:", MODEL_REVISION)

In [ ]:
from datasets import load_dataset

data = load_dataset("json", data_files={"train": f"{DATA_DIR}/train.jsonl",
                                        "validation": f"{DATA_DIR}/validation.jsonl"})
data = data.select_columns(["prompt", "completion"])   # metadata stays in the files, not in training
lengths = [len(tokenizer.apply_chat_template(r["prompt"] + r["completion"], tokenize=True))
           for split in data.values() for r in split]
MAX_TOKENS = max(lengths)
assert MAX_TOKENS <= tr.TRAINER_CONFIG["max_length"], f"{MAX_TOKENS} tokens would be truncated"
print(data)
print("longest prompt + completion:", MAX_TOKENS, "tokens")

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

args = SFTConfig(output_dir=f"{RUN_DIR}/checkpoints", **tr.TRAINER_CONFIG)
trainer = SFTTrainer(model=model, args=args, train_dataset=data["train"], eval_dataset=data["validation"],
                     processing_class=tokenizer, peft_config=LoraConfig(**tr.LORA_CONFIG))
assert "completion_mask" in trainer.train_dataset.column_names, "loss would not be completion-only"
TRAINABLE = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
trainer.model.print_trainable_parameters()

In [ ]:
import math

STARTED_AT = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
start = time.time()
trainer.train()
RUNTIME = round(time.time() - start, 1)

LOSSES = tr.epoch_losses(trainer.state.log_history)
for row in LOSSES:
    print(row)
bad = [e for e in LOSSES if not (isinstance(e["eval_loss"], float) and math.isfinite(e["eval_loss"]))]
assert not bad, f"missing or non-finite validation loss: {bad}"
print("kept:", trainer.state.best_model_checkpoint, "eval_loss", trainer.state.best_metric, "| runtime", RUNTIME, "s")

`load_best_model_at_end` has already put the lowest-validation-loss checkpoint back into `trainer.model`; save that adapter.

In [ ]:
trainer.save_model(ADAPTER_DIR)
print("adapter saved to Drive:", ADAPTER_DIR)

In [ ]:
import importlib.metadata as md
import json
import platform

def version(package):
    try:
        return md.version(package)
    except md.PackageNotFoundError:
        return None

versions = {p: version(p) for p in ("torch", "transformers", "trl", "peft", "accelerate", "datasets")}
versions["python"] = platform.python_version()

record = tr.run_record(
    manifest_sha256=MANIFEST_SHA256, dataset_folder=DATASET_FOLDER, versions=versions, gpu=GPU,
    runtime_seconds=RUNTIME, log_history=trainer.state.log_history,
    best_checkpoint=trainer.state.best_model_checkpoint, best_metric=trainer.state.best_metric,
    adapter_location=ADAPTER_DIR,
    extra={"run_id": RUN_ID, "started_at": STARTED_AT, "model_revision": MODEL_REVISION, "cuda": torch.version.cuda,
           "train_rows": len(data["train"]), "validation_rows": len(data["validation"]),
           "max_prompt_completion_tokens": MAX_TOKENS, "trainable_parameters": TRAINABLE,
           "pinned_packages": list(tr.PINNED_PACKAGES), "trainer_args_full": json.loads(args.to_json_string())})
tr.write_run_record(f"{RUN_DIR}/run_record.json", record)
files.download(f"{RUN_DIR}/run_record.json")

**After the run:** save the downloaded `run_record.json` as
`component3_socratic_educator/evidence/inquire_training_<date>/run_record.json` in the repo.
The adapter stays on Drive (`ADAPTER_DIR`); ticket 06 loads it from there for the comparison.